# Data Reading

In [0]:
df = spark.read.format("parquet")\
            .option('inferSchema',True)\
            .load('abfss://bronze@carshrudatalake.dfs.core.windows.net/rawdata')
df.display()
  

# DATA TRANSFORMATION

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
df = df.withColumn('model_category',split(col('Model_ID'),'-')[0])
df.display()

In [0]:
df.withColumn('Units_sold', col('Units_Sold').cast(StringType())).printSchema()

In [0]:
df = df.withColumn('RevPerUnit',col('Revenue')/col('Units_Sold'))

df.display()

# AD-HOC    

In [0]:
display(df.groupBy('BranchName').agg(sum('Units_Sold').alias('Total_Units')).sort('Total_Units'))

Databricks visualization. Run in Databricks to view.

## DATA WRITING

In [0]:
df.write.format('parquet')\
    .mode('overwrite')\
    .option('path','abfss://silver@carshrudatalake.dfs.core.windows.net/carsales')\
    .save()

## Querying Silver Data

In [0]:
%sql

SELECT * FROM parquet.`abfss://silver@carshrudatalake.dfs.core.windows.net/carsales`